# Imports

In [1]:
import optuna
import pickle

import numpy as np
import pandas as pd

from utils import load_pickle

from sklearn.metrics import balanced_accuracy_score
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import StratifiedKFold, cross_val_predict

/home/junior/Documentos/GitHub/kaggle-competition-predicting-stellar-class/.venv/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## Utils

In [2]:
label_encoder = load_pickle('../models/label_encoder.pkl')

# Loading Datasets

In [3]:
X_train = pd.read_parquet('../data/X_train_stacking_layer_three.parquet')
y_train = pd.read_parquet('../data/y_train.parquet')

X_test = pd.read_parquet('../data/X_test_stacking_layer_three.parquet')

In [4]:
X_train.head()

,lgbm_0,lgbm_1,lgbm_2,cat_0,cat_1,cat_2,xgb_0,xgb_1,xgb_2,hist_0,hist_1,hist_2,rf_0,rf_1,rf_2,extra_0,extra_1,extra_2
0,0.999778,0.000212,0.000009,0.999787,0.000174,0.000040,0.999864,0.000121,0.000016,0.999733,0.000264,2.459617e-06,0.999775,0.000225,2.616556e-08,0.998912,0.000955,0.000133
1,0.994627,0.000176,0.005197,0.995247,0.000365,0.004388,0.992235,0.000199,0.007566,0.973059,0.000137,2.680453e-02,0.994737,0.000047,5.216709e-03,0.976808,0.000850,0.022342
2,0.000048,0.999944,0.000008,0.000483,0.999485,0.000032,0.000169,0.999822,0.000009,0.000015,0.999984,1.319155e-06,0.000011,0.999989,9.836632e-08,0.000068,0.999897,0.000035
3,0.999915,0.000081,0.000005,0.999745,0.000217,0.000039,0.999858,0.000127,0.000014,0.999960,0.000040,6.602657e-07,0.999793,0.000207,3.828407e-08,0.998919,0.000947,0.000134
4,0.998164,0.001812,0.000024,0.998143,0.001764,0.000093,0.998504,0.001459,0.000037,0.986426,0.013556,1.806930e-05,0.998227,0.001769,3.566911e-06,0.994430,0.005248,0.000322


In [5]:
X_test.head()

,lgbm_0,lgbm_1,lgbm_2,cat_0,cat_1,cat_2,xgb_0,xgb_1,xgb_2,hist_0,hist_1,hist_2,rf_0,rf_1,rf_2,extra_0,extra_1,extra_2
0,0.998177,0.001784,0.000039,0.997654,0.002153,0.000193,0.998108,0.001843,0.000049,0.997980,0.002008,0.000012,0.998183,0.001800,0.000017,0.993598,0.005832,0.000569
1,0.997269,0.002710,0.000021,0.997945,0.001978,0.000077,0.997185,0.002778,0.000037,0.994660,0.005324,0.000015,0.997905,0.002092,0.000003,0.993669,0.005970,0.000361
2,0.996089,0.000609,0.003303,0.997698,0.000711,0.001592,0.997502,0.000382,0.002115,0.997295,0.001499,0.001207,0.997506,0.000715,0.001780,0.988872,0.002584,0.008544
3,0.000643,0.000107,0.999250,0.001658,0.000165,0.998177,0.000730,0.000132,0.999138,0.000139,0.000020,0.999841,0.000541,0.000023,0.999436,0.000378,0.000214,0.999409
4,0.999631,0.000357,0.000012,0.999578,0.000365,0.000058,0.999737,0.000245,0.000018,0.999526,0.000469,0.000005,0.999785,0.000205,0.000010,0.998670,0.001116,0.000214


# Machine Learning

In [6]:
def objective(trial, X, y):

    cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

    scores = []

    for fold, (train_idx, valid_idx) in enumerate(cv.split(X, y)):

        X_train_fold = X.iloc[train_idx, :]
        X_valid_fold = X.iloc[valid_idx, :]

        y_train_fold = y.iloc[train_idx]
        y_valid_fold = y.iloc[valid_idx]

        model = LogisticRegression(
            solver=trial.suggest_categorical("solver", ["saga"]),
            C=trial.suggest_float("C", 1e-5, 100, log=True),
            l1_ratio=trial.suggest_float("l1_ratio", 0.0, 1.0),
            class_weight=trial.suggest_categorical("class_weight", [None, "balanced"]),
            fit_intercept=trial.suggest_categorical("fit_intercept", [True, False]),
            tol=trial.suggest_float("tol", 1e-6, 1e-2, log=True),
            max_iter=trial.suggest_int("max_iter", 2000, 5000),
        ).fit(X_train_fold, y_train_fold)

        proba = model.predict_proba(X_valid_fold)

        w0 = trial.suggest_float('weight_class_0', 0.1, 100.0)
        w1 = trial.suggest_float('weight_class_1', 0.1, 100.0)
        w2 = trial.suggest_float('weight_class_2', 0.1, 100.0)

        weights = np.array([w0, w1, w2])
        weighted_probas = proba * weights

        pred = np.argmax(weighted_probas, axis=1)
        
        score = balanced_accuracy_score(y_valid_fold, pred)
        scores.append(score)

        trial.report(np.mean(scores), step=fold)

        if trial.should_prune():
            raise optuna.exceptions.TrialPruned()

    return np.mean(scores)

study = optuna.create_study(direction="maximize", sampler=optuna.samplers.TPESampler(seed=42), pruner=optuna.pruners.MedianPruner(n_warmup_steps=2))
study.optimize(lambda trial: objective(trial, X_train, y_train.class_encoded), n_trials=30, n_jobs=-1, show_progress_bar=True)


print("Best trial score:")
print(study.best_trial.value)

print("\nBest params:")
print(study.best_trial.params)

[I 2026-06-18 10:22:38,915] A new study created in memory with name: no-name-c8d4e97e-abe1-4feb-941d-cd25a65fefa6
Best trial: 4. Best value: 0.96369:   3%|████▋                                                                                                                                       | 1/30 [00:36<17:42, 36.64s/it]

[I 2026-06-18 10:23:15,546] Trial 4 finished with value: 0.9636904475990885 and parameters: {'solver': 'saga', 'C': 0.025116306458195778, 'l1_ratio': 0.13447175935836297, 'class_weight': 'balanced', 'fit_intercept': False, 'tol': 0.00031867477614607576, 'max_iter': 3077, 'weight_class_0': 64.64507909318253, 'weight_class_1': 24.005777859025873, 'weight_class_2': 19.53246691220257}. Best is trial 4 with value: 0.9636904475990885.


Best trial: 4. Best value: 0.96369:   7%|█████████▎                                                                                                                                  | 2/30 [00:38<07:36, 16.31s/it]

[I 2026-06-18 10:23:17,597] Trial 2 finished with value: 0.962637073111356 and parameters: {'solver': 'saga', 'C': 0.00036814457985981424, 'l1_ratio': 0.89773753118166, 'class_weight': 'balanced', 'fit_intercept': True, 'tol': 0.00010326173696425072, 'max_iter': 3343, 'weight_class_0': 74.78742218585307, 'weight_class_1': 45.66362965735184, 'weight_class_2': 22.23358930906439}. Best is trial 4 with value: 0.9636904475990885.


Best trial: 4. Best value: 0.96369:  10%|██████████████                                                                                                                              | 3/30 [00:39<04:10,  9.26s/it]

...

[I 2026-06-18 10:27:49,103] Trial 18 pruned. 


Best trial: 27. Best value: 0.966384:  97%|████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████▍    | 29/30 [05:55<00:47, 47.96s/it]

[I 2026-06-18 10:28:34,127] Trial 23 pruned. 


Best trial: 27. Best value: 0.966384: 100%|█████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 30/30 [12:58<00:00, 25.94s/it]

[I 2026-06-18 10:35:37,183] Trial 10 pruned. 
Best trial score:
0.966384056041312

Best params:
{'solver': 'saga', 'C': 2.6336088899421295, 'l1_ratio': 0.5567252200901791, 'class_weight': 'balanced', 'fit_intercept': True, 'tol': 0.005507757703013647, 'max_iter': 2275, 'weight_class_0': 86.92317027715983, 'weight_class_1': 97.87994073895375, 'weight_class_2': 97.27761584260433}


In [18]:
study.optimize(lambda trial: objective(trial, X_train, y_train.class_encoded), n_trials=30, n_jobs=-1, show_progress_bar=True)

Best trial: 59. Best value: 0.966501:   3%|████▌                                                                                                                                     | 1/30 [00:18<09:05, 18.81s/it]

[I 2026-06-18 10:55:52,024] Trial 60 pruned. 


Best trial: 59. Best value: 0.966501:   7%|█████████▏                                                                                                                                | 2/30 [00:19<03:46,  8.09s/it]

[I 2026-06-18 10:55:52,600] Trial 61 pruned. 


Best trial: 59. Best value: 0.966501:  13%|██████████████████▍                                                                                                                       | 4/30 [00:20<01:14,  2.88s/it]

...

[I 2026-06-18 11:00:10,958] Trial 77 finished with value: 0.9664506363627854 and parameters: {'solver': 'saga', 'C': 0.9731747259754717, 'l1_ratio': 0.8502059776949186, 'class_weight': 'balanced', 'fit_intercept': True, 'tol': 0.00014625376732145317, 'max_iter': 4405, 'weight_class_0': 48.26990230996273, 'weight_class_1': 92.57128981026096, 'weight_class_2': 92.70860346031922}. Best is trial 59 with value: 0.9665009397831099.


Best trial: 59. Best value: 0.966501:  97%|████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████▍    | 29/30 [04:42<00:04,  4.34s/it]

[I 2026-06-18 11:00:15,524] Trial 74 finished with value: 0.9664473654585322 and parameters: {'solver': 'saga', 'C': 0.8982105322796461, 'l1_ratio': 0.8742170780053262, 'class_weight': 'balanced', 'fit_intercept': True, 'tol': 0.00011489101523439219, 'max_iter': 4344, 'weight_class_0': 54.089599821611074, 'weight_class_1': 92.4173342512778, 'weight_class_2': 92.85661410372009}. Best is trial 59 with value: 0.9665009397831099.


Best trial: 59. Best value: 0.966501: 100%|█████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 30/30 [05:13<00:00, 10.46s/it]

[I 2026-06-18 11:00:47,063] Trial 86 finished with value: 0.9664379407639793 and parameters: {'solver': 'saga', 'C': 0.733133322933437, 'l1_ratio': 0.7205834099572869, 'class_weight': 'balanced', 'fit_intercept': True, 'tol': 4.3928908646947333e-05, 'max_iter': 4591, 'weight_class_0': 50.54513895621818, 'weight_class_1': 91.6536723519764, 'weight_class_2': 86.69815873652722}. Best is trial 59 with value: 0.9665009397831099.


In [19]:
lg_params = {k: v for k, v in study.best_params.items() if k not in ['weight_class_0', 'weight_class_1', 'weight_class_2']}

lg = LogisticRegression(**lg_params).fit(X_train, y_train.class_encoded)

test_proba = lg.predict_proba(X_test)

weights = np.array([study.best_params['weight_class_0'], study.best_params['weight_class_1'], study.best_params['weight_class_2']])
weighted_probas = test_proba * weights

pred = np.argmax(weighted_probas, axis=1)

In [20]:
sub_labels = label_encoder.inverse_transform(pred)

# Submission

In [21]:
submission = pd.read_csv('../data/sample_submission.csv')
submission['class'] = sub_labels

submission.to_csv('../data/submission_stacking_lg.csv', index=False)

In [22]:
submission.head()

,id,class
0,577347,GALAXY
1,577348,GALAXY
2,577349,GALAXY
3,577350,STAR
4,577351,GALAXY


In [23]:
X_train.columns

Index(['lgbm_0', 'lgbm_1', 'lgbm_2', 'cat_0', 'cat_1', 'cat_2', 'xgb_0',
       'xgb_1', 'xgb_2', 'hist_0', 'hist_1', 'hist_2', 'rf_0', 'rf_1', 'rf_2',
       'extra_0', 'extra_1', 'extra_2'],
      dtype='str')